In [1]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import pathlib

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "india"
vehicle = "rice"
scenario = "intervention"

In [3]:
# Parameters
location = "ethiopia"
vehicle = "salt"
scenario = "intervention_100_nrv"


In [4]:
def aggregate_by_scenario(df):
    return df.groupby(["scenario", "input_draw", "wealth_quintile"]).value.sum().groupby(["scenario", "wealth_quintile"]).mean()

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if pathlib.Path(path).is_file():
    pregnancy_person_time_anemia = (
        pd.read_parquet(path)
    )
else:
    pregnancy_person_time_anemia = (
        pd.read_parquet(f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,lowest,baseline,50,0,0
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,second,baseline,50,0,0
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,middle,baseline,50,0,0
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,fourth,baseline,50,0,0
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,highest,baseline,50,0,0
...,...,...,...,...,...,...,...,...,...,...,...
719995,person_time,impairment,anemia,severe,95_plus,severe,lowest,intervention_100_nrv,144,0,0
719996,person_time,impairment,anemia,severe,95_plus,severe,second,intervention_100_nrv,144,0,0
719997,person_time,impairment,anemia,severe,95_plus,severe,middle,intervention_100_nrv,144,0,0
719998,person_time,impairment,anemia,severe,95_plus,severe,fourth,intervention_100_nrv,144,0,0


In [6]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline                200
intervention_100_nrv    200
Name: random_seed, dtype: int64

In [7]:
pregnancy_person_time_anemia.sub_entity.value_counts()

mild          180000
moderate      180000
not_anemic    180000
severe        180000
Name: sub_entity, dtype: int64

In [8]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario              wealth_quintile
baseline              fourth             0.0
                      highest            0.0
                      lowest             0.0
                      middle             0.0
                      second             0.0
intervention_100_nrv  fourth             0.0
                      highest            0.0
                      lowest             0.0
                      middle             0.0
                      second             0.0
Name: value, dtype: float64

In [9]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[pregnancy_person_time_anemia.sub_entity != 'not_anemic']
)
anemic_pregnant_person_time

scenario              wealth_quintile
baseline              fourth             0.0
                      highest            0.0
                      lowest             0.0
                      middle             0.0
                      second             0.0
intervention_100_nrv  fourth             0.0
                      highest            0.0
                      lowest             0.0
                      middle             0.0
                      second             0.0
Name: value, dtype: float64

In [10]:
pregnant_anemia_prevalence_by_scenario = (anemic_pregnant_person_time / total_pregnant_person_time).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario              wealth_quintile
baseline              fourth             0.0
                      highest            0.0
                      lowest             0.0
                      middle             0.0
                      second             0.0
intervention_100_nrv  fourth             0.0
                      highest            0.0
                      lowest             0.0
                      middle             0.0
                      second             0.0
Name: value, dtype: float64

In [11]:
path = f'./results/{location}/{vehicle}/{scenario}/pregnant_anemia_prevalence_by_scenario.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [12]:
pop = pd.read_csv(f'../0100_data_prep/results/population/stratified/{location}.csv')
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,lowest,7729.330504
1,Female,0.0,0.019178,not_pregnant,second,7026.085890
2,Female,0.0,0.019178,not_pregnant,middle,6642.401601
3,Female,0.0,0.019178,not_pregnant,fourth,5956.884779
4,Female,0.0,0.019178,not_pregnant,highest,4774.758589
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,lowest,659.578610
281,Male,95.0,125.000000,not_pregnant,second,690.099645
282,Male,95.0,125.000000,not_pregnant,middle,716.289850
283,Male,95.0,125.000000,not_pregnant,fourth,757.897451


In [13]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
fourth     643787.343974
highest    642344.561183
lowest     831426.672018
middle     692466.247031
second     844624.021487
Name: value, dtype: float64

In [14]:
pregnancy_prevalent_anemia_cases_by_scenario = pregnant_anemia_prevalence_by_scenario * pregnant_pop
pregnancy_prevalent_anemia_cases_by_scenario

scenario              wealth_quintile
baseline              fourth             0.0
                      highest            0.0
                      lowest             0.0
                      middle             0.0
                      second             0.0
intervention_100_nrv  fourth             0.0
                      highest            0.0
                      lowest             0.0
                      middle             0.0
                      second             0.0
Name: value, dtype: float64

In [15]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if pathlib.Path(path).is_file():
    maternal_disorders_transition_counts = (
        pd.read_parquet(path)
    )
else:
    maternal_disorders_transition_counts = (
        pd.read_parquet(f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,lowest,baseline,50,0,0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,second,baseline,50,0,0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,middle,baseline,50,0,0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,fourth,baseline,50,0,0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,highest,baseline,50,0,0
...,...,...,...,...,...,...,...,...,...,...,...
359995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,lowest,intervention_100_nrv,144,0,0
359996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,second,intervention_100_nrv,144,0,0
359997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,middle,intervention_100_nrv,144,0,0
359998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,fourth,intervention_100_nrv,144,0,0


In [16]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['maternal_disorders_to_recovered_from_maternal_disorders',
       'no_transition',
       'susceptible_to_maternal_disorders_to_maternal_disorders'],
      dtype='object')

In [17]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[maternal_disorders_transition_counts.sub_entity == 'susceptible_to_maternal_disorders_to_maternal_disorders']
)
maternal_disorders_incident_cases_by_scenario

scenario              wealth_quintile
baseline              fourth             0.0
                      highest            0.0
                      lowest             0.0
                      middle             0.0
                      second             0.0
intervention_100_nrv  fourth             0.0
                      highest            0.0
                      lowest             0.0
                      middle             0.0
                      second             0.0
Name: value, dtype: float64

In [18]:
path = f'./results/{location}/{vehicle}/{scenario}/maternal_disorders_incident_cases_by_scenario.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [19]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"
if pathlib.Path(path).is_file():
    neonatal_deaths = (
        pd.read_parquet(path)
    )
else:
    neonatal_deaths = (
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet").assign(value=0).assign(maternal_scenario=lambda x: x.maternal_scenario.replace('intervention', scenario))
    )

neonatal_deaths = neonatal_deaths.rename(columns={"maternal_scenario": "scenario"})
neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,random_seed,input_draw,value
0,deaths,cause,stillborn,stillborn,0_to_6_months,Female,lowest,baseline,baseline,69,0,0
1,deaths,cause,stillborn,stillborn,0_to_6_months,Female,second,baseline,baseline,69,0,0
2,deaths,cause,stillborn,stillborn,0_to_6_months,Female,middle,baseline,baseline,69,0,0
3,deaths,cause,stillborn,stillborn,0_to_6_months,Female,fourth,baseline,baseline,69,0,0
4,deaths,cause,stillborn,stillborn,0_to_6_months,Female,highest,baseline,baseline,69,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
31995,deaths,cause,other_causes,other_causes,18_to_59_months,Male,lowest,baseline,baseline,96,0,0
31996,deaths,cause,other_causes,other_causes,18_to_59_months,Male,second,baseline,baseline,96,0,0
31997,deaths,cause,other_causes,other_causes,18_to_59_months,Male,middle,baseline,baseline,96,0,0
31998,deaths,cause,other_causes,other_causes,18_to_59_months,Male,fourth,baseline,baseline,96,0,0


In [20]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario              wealth_quintile
baseline              fourth             0.0
                      highest            0.0
                      lowest             0.0
                      middle             0.0
                      second             0.0
intervention_100_nrv  fourth             0.0
                      highest            0.0
                      lowest             0.0
                      middle             0.0
                      second             0.0
Name: value, dtype: float64

In [21]:
path = f'./results/{location}/{vehicle}/{scenario}/neonatal_deaths_by_scenario.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [22]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/{scenario}/anemia_cases.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_cases = (
        pd.read_parquet(path)
    )
else:
    non_pregnancy_anemia_cases = (
        pd.read_parquet(f"../0400_non_pregnant_anemia_model/results/rice/india/intervention/anemia_cases.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,fourth,0,baseline
1,Female,0.0,0.019178,highest,0,baseline
2,Female,0.0,0.019178,lowest,0,baseline
3,Female,0.0,0.019178,middle,0,baseline
4,Female,0.0,0.019178,second,0,baseline
...,...,...,...,...,...,...
495,Male,95.0,125.000000,fourth,0,intervention_100_nrv
496,Male,95.0,125.000000,highest,0,intervention_100_nrv
497,Male,95.0,125.000000,lowest,0,intervention_100_nrv
498,Male,95.0,125.000000,middle,0,intervention_100_nrv


In [23]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0"))
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario              wealth_quintile
baseline              fourth             0.0
                      highest            0.0
                      lowest             0.0
                      middle             0.0
                      second             0.0
intervention_100_nrv  fourth             0.0
                      highest            0.0
                      lowest             0.0
                      middle             0.0
                      second             0.0
Name: value, dtype: float64

In [24]:
prevalent_anemia_cases_by_scenario = pregnancy_prevalent_anemia_cases_by_scenario + non_pregnancy_prevalent_anemia_cases_by_scenario
prevalent_anemia_cases_by_scenario

scenario              wealth_quintile
baseline              fourth             0.0
                      highest            0.0
                      lowest             0.0
                      middle             0.0
                      second             0.0
intervention_100_nrv  fourth             0.0
                      highest            0.0
                      lowest             0.0
                      middle             0.0
                      second             0.0
Name: value, dtype: float64

In [25]:
path = f'./results/{location}/{vehicle}/{scenario}/prevalent_anemia_cases_by_scenario.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [26]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/{scenario}/ntd_cases_by_scenario.csv"
if pathlib.Path(path).is_file():
    ntd_cases_by_scenario = (
        pd.read_csv(path)
    )
else:
    ntd_cases_by_scenario = (
        pd.read_csv(f"../0500_neural_tube_defects_model/results/india/rice/intervention/ntd_cases_by_scenario.csv").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(["scenario", "wealth_quintile"]).value
ntd_cases_by_scenario

scenario              wealth_quintile
baseline              lowest             1756.556323
                      second             1670.886846
                      middle             1520.799236
                      fourth             1340.162820
                      highest            1057.254316
intervention_100_nrv  fourth              591.747312
                      highest             555.684528
                      lowest              369.498415
                      middle              303.672729
                      second              381.991736
Name: value, dtype: float64

In [27]:
path = f'./results/{location}/{vehicle}/{scenario}/ntd_cases_by_scenario.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)